# ROPG-KD Retriever Training

Stage 1 of Simurgh's two-stage training: LoRA fine-tuning of the Qwen3-Embedding-0.6B
encoder, distilling the offline LLM-judge teacher scores in `data/ropg_kd/{train,val}.jsonl`
(see `notebooks/gen_ropg_data.ipynb`). Checkpoints are selected on validation **Recall@K
over the full corpus, reported per persona** — not on KD loss.

The trainer cell below is a **verbatim copy of `src/rl/ropg_kd.py`**, generated by
`notebooks/build_train_ropg_kd.py`. Do not hand-edit it: change `src/rl/ropg_kd.py` and
regenerate, or the two will drift the way they did before.

**Multi-GPU.** The launch cell uses `torchrun`, which starts one independent OS process
per GPU. Two consequences shape the cells below:
1. torchrun executes a **script path**, so the trainer is written to `ropg_kd_worker.py`
   by the `%%writefile` cell, and `CFG` is dumped to YAML — a fresh interpreter cannot
   receive a Python dict.
2. A fresh interpreter cannot be handed a live Python dict, so `CFG` is written out as
   YAML and passed with `--config`. That also leaves each run reproducible from an
   artifact rather than from notebook state.

**Kaggle setup**
1. Enable GPU accelerator (T4 x2 for DDP; T4 x1 also works).
2. Enable internet access (the encoder is downloaded from Hugging Face).
3. Attach the `simurgh-data` dataset, **version 2+** (must contain `ropg_kd/` and `chunks/corpus.jsonl`).
4. No API secrets needed - training makes no LLM calls.

**Colab setup**
1. Set `RUNTIME = "colab"` in the Config cell.
2. Upload `simurgh-data/` to Drive at `MyDrive/simurgh-data/` with `ropg_kd/{train,val}.jsonl`
   and `chunks/corpus.jsonl`.
3. Enable GPU accelerator. Checkpoints save straight to Drive, so the zip cell is skipped.


In [ ]:
!pip install -q -U transformers accelerate peft

# peft's LoRA injection probes every dispatcher, including torchao, for every target
# module. Kaggle's image ships torchao 0.10.0 and recent peft rejects anything below
# 0.16.0 with an ImportError rather than skipping it - so adapter creation dies even
# though nothing here is quantized. Removing the package makes peft's find_spec check
# return False cleanly; upgrading it instead would pull torchao's own torch pin and
# risk replacing Kaggle's CUDA-matched torch build.
!pip uninstall -y -q torchao

## Config

Mirrors `configs/train_ropg.yaml`; the last cell warns if the two disagree.

In [ ]:
import os

# Must be set before torch initializes CUDA; mitigates fragmentation OOMs by letting
# the allocator grow segments instead of hunting for contiguous blocks.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from pathlib import Path

# -- Runtime selector ---------------------------------------------------------
RUNTIME = "kaggle"  # "kaggle" | "colab" | "local"
GDRIVE_BASE = "/content/drive/MyDrive/simurgh-data"  # Colab only

if RUNTIME == "kaggle":
    DATA_ROOT = "/kaggle/input/datasets/alirezahsn/simurgh-data"
    OUTPUT_DIR = "/kaggle/working/ropg_kd_checkpoints"
    WORKDIR = "/kaggle/working"
elif RUNTIME == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_ROOT = GDRIVE_BASE
    OUTPUT_DIR = f"{GDRIVE_BASE}/ropg_kd_checkpoints"
    WORKDIR = "/content"
else:  # local
    DATA_ROOT = "data"
    OUTPUT_DIR = "data/rl/ropg_kd_checkpoints"
    WORKDIR = "."

# -- Inline config (mirrors configs/train_ropg.yaml) ---------------------------
CFG = {
    "seed": 42,
    "mode": "reader_kd",   # hard_neg | reader_kd
    "format": "triplets",  # only used when mode=hard_neg
    "data": {
        "train_data": f"{DATA_ROOT}/ropg_kd",
        "corpus_path": f"{DATA_ROOT}/chunks/corpus.jsonl",
    },
    "embedder": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        # Must match the length used when the FAISS index is built, or the same chunk
        # gets two different embeddings at train and serve time. At 2048 the p95 corpus
        # chunk fits whole; only the longest few still truncate.
        "max_seq_length": 2048,
    },
    "training": {
        "device": "cuda",
        "precision": "fp16",   # T4 has no bf16; use bf16 only on A100+
        # Not a tunable on a 16 GB T4: without it one batch_size:1 step retains ~36 GB
        # of activations, with it ~2.5 GB (only each block's input is stored). No
        # batch_size/doc_micro_batch pair fits without it; it costs one extra forward.
        "gradient_checkpointing": True,
        # Mirrors the YAML. Watch the wall clock before raising it: the last 2xT4 run took
        # ~1h52m per epoch, so 5 epochs is ~9.3h and will not finish inside a Kaggle
        # session. Lower it *here and in the YAML together* - the drift check compares them.
        "epochs": 5,
        # groups per step; effective batch = this x n_gpus. The knob for *retained*
        # activation memory: one graph is held until backward, so stored activations
        # scale with tokens per step and only this controls them.
        "batch_size": 2,
        # sequences per forward inside encode_chunked. Caps the *transient* per-forward
        # working set (all micro-forwards stay in one graph, so it does not reduce
        # retained activations) AND sets padding tightness: encode_chunked sorts by true
        # length and trims each micro-batch to its own width. The latter dominates - the
        # median document is 167 tokens against a 2048 batch-wide pad, so unsorted
        # chunking wastes ~65% of every FLOP. Measured: 20 -> 1.68x, 10 -> 2.21x,
        # 5 -> 2.54x; 10 takes nearly all of it without starving the GPU. 0 disables.
        "doc_micro_batch": 10,
        "lr": 2.0e-4,          # LoRA rate - a full-FT rate (2e-5) under-trains an adapter
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "grad_clip": 1.0,
        "num_workers": 2,
        # Cosine sims live in [-1,1] and teacher scores in [0,1]; at T=1.0 a softmax
        # over 20 candidates is near-uniform on both sides and the gradient is noise.
        "student_temp": 0.05,
        "teacher_temp": 0.2,
        "max_negatives": 4,
        "max_documents": 20,
    },
    "lora": {
        "r": 8,
        "alpha": 16,
        "dropout": 0.1,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    "eval": {
        "top_k": 5,
        "relevance_top_m": 3,  # relevant = top-m docs by teacher_score per group
        # no-grad corpus + query encoding, so unconstrained by training batch_size;
        # rank 0 only, once per epoch.
        "eval_batch_size": 32,
    },
    "checkpoint_dir": OUTPUT_DIR,
}

Path(CFG["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

# torchrun launches a fresh interpreter, which can only be handed a *file*, not a
# Python dict. Writing CFG out also leaves the run reproducible from an artifact.
import yaml

CFG_PATH = str(Path(WORKDIR) / "train_ropg_generated.yaml")
Path(CFG_PATH).write_text(yaml.safe_dump(CFG, sort_keys=False), encoding="utf-8")
print("Config ready. cwd:", os.getcwd(), "| config:", CFG_PATH)


## Trainer

Verbatim copy of `src/rl/ropg_kd.py`, written to disk so `torchrun` can execute it.
Generated - edit the source module and re-run `notebooks/build_train_ropg_kd.py` instead.

In [ ]:
%%writefile ropg_kd_worker.py
"""ROPG-KD: fine-tune the Qwen3-Embedding-0.6B encoder via knowledge distillation from LLM judge scores.

Stage 1 of Simurgh's two-stage training:
  1. Load (query, persona, docs) groups scored by the offline LLM judge
     (notebooks/gen_ropg_data.ipynb, mirroring configs/datagen_ropg.yaml),
     each doc carrying a teacher_score in [0, 1].
  2. Fine-tune a LoRA adapter over Qwen3-Embedding-0.6B to minimise the listwise
     KL divergence between its similarity distribution over the group's docs and the
     teacher's softmax utility distribution.
  3. Validate every epoch with Recall@K / MRR **per persona over the full corpus**
     and checkpoint on Recall@K (ties: lower val KL, then higher MRR).

Supports two training modes:
  - hard_neg: Multiple Negatives Ranking Loss (MNRL) — ablation arm
  - reader_kd: KL-distillation against continuous reader scores — primary

The encoder here MUST stay byte-for-byte equivalent to inference
(``rag.embedder.Qwen3Embedder``): last-token pooling, documents encoded bare, and
queries wrapped as ``Instruct: {persona}\\nQuery: {text}``. ``tests/test_encoder_parity.py``
is the gate on that; a pooling or prompt mismatch silently destroys the fine-tune.

Runs on GPU (Kaggle / university cluster). Install the training extras first:
    uv sync --extra embedding --extra training

Multi-GPU is launched exclusively with ``torchrun``; there is no in-process
spawning. ``main_worker`` reads ``LOCAL_RANK``/``RANK``/``WORLD_SIZE`` from the
environment, which torchrun sets and which default to a single process otherwise.

Typical usage:
    uv run python -m rl.ropg_kd --config configs/train_ropg.yaml
    torchrun --standalone --nproc_per_node=2 -m rl.ropg_kd --config configs/train_ropg.yaml
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import random
from datetime import timedelta
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
import yaml
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset, DistributedSampler
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

# --- inlined from src/personalization/profiles.py (no repo install on Kaggle) ---
from dataclasses import dataclass


@dataclass(frozen=True)
class Profile:
    id: str
    comprehension: str  # L1–L4
    prior_knowledge: str  # L1–L4
    learning_goal: str  # L1–L4
    explanation_style: str  # L1–L4
    split: str  # "train" or "test"
    rendered: str  # natural-language prose injected into prompts


PERSONAS: dict[str, Profile] = {
    "crammer": Profile(
        id="crammer",
        comprehension="L1",
        prior_knowledge="L1",
        learning_goal="L1",
        explanation_style="L4",
        split="train",
        rendered=(
            "A ninth-grader who finds the textbook hard to follow and has little background "
            "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
            "to score — but it must be spelled out simply, step by step, with examples."
        ),
    ),
    "scholar": Profile(
        id="scholar",
        comprehension="L4",
        prior_knowledge="L3",
        learning_goal="L4",
        explanation_style="L1",
        split="train",
        rendered=(
            "A ninth-grader who reads dense material easily and has solid background on this "
            "topic. Wants to understand the underlying why and how, and the connections between "
            "ideas. Prefers a terse, high-level treatment without hand-holding or padding."
        ),
    ),
    "steady": Profile(
        id="steady",
        comprehension="L3",
        prior_knowledge="L2",
        learning_goal="L2",
        explanation_style="L2",
        split="train",
        rendered=(
            "A capable ninth-grader with average background on this topic. Wants a correct "
            "answer with a brief justification, balanced toward exam needs. Does not need "
            "elaborate scaffolding, but does appreciate a one-line reason."
        ),
    ),
    "newcomer": Profile(
        id="newcomer",
        comprehension="L3",
        prior_knowledge="L1",
        learning_goal="L4",
        explanation_style="L4",
        split="test",
        rendered=(
            "A bright ninth-grader who reads well but is new to this topic. Wants real "
            "understanding — the why and how, not just the answer — and needs worked examples "
            "to bridge the missing background."
        ),
    ),
}


def render_profile(persona_id: str) -> str:
    """Return the natural-language rendering for *persona_id*."""
    if persona_id not in PERSONAS:
        raise ValueError(f"Unknown persona: {persona_id!r}. Valid: {list(PERSONAS)}")
    return PERSONAS[persona_id].rendered


def train_personas() -> list[Profile]:
    """Return the three training personas (excludes the test holdout)."""
    return [p for p in PERSONAS.values() if p.split == "train"]
# --- end inlined profiles ---

# --- inlined from src/data/gen_ropg_data.py (no repo install on Kaggle) ---
def derive_triplets(scored_path: Path, output_dir: Path, max_negatives: int = 4) -> None:
    """Derive hard-negative triplets and pairs from a scored JSONL file.

    Writes two files alongside the scored file:
      {stem}_triplets.jsonl — one line per group: {query, persona_id, positive, negatives:[...]}
      {stem}_pairs.jsonl    — one line per (pos, neg) pair: {query, persona_id, positive, negative}
    """
    stem = scored_path.stem          # e.g. "train" or "val"
    triplet_path = output_dir / f"{stem}_triplets.jsonl"
    pairs_path   = output_dir / f"{stem}_pairs.jsonl"

    n_triplets = 0
    n_pairs    = 0

    with (
        triplet_path.open("w", encoding="utf-8") as tf,
        pairs_path.open("w", encoding="utf-8")   as pf,
    ):
        for raw in scored_path.read_text(encoding="utf-8").splitlines():
            if not raw.strip():
                continue
            rec = json.loads(raw)
            docs = rec.get("docs", [])
            if len(docs) < 2:
                continue

            sorted_docs = sorted(docs, key=lambda d: d["teacher_score"], reverse=True)
            positive    = sorted_docs[0]["text"]
            negatives   = [d["text"] for d in sorted_docs[-max_negatives:]]

            base = {"query": rec["query"], "persona_id": rec.get("persona_id", "")}

            tf.write(json.dumps({**base, "positive": positive, "negatives": negatives}, ensure_ascii=False) + "\n")
            n_triplets += 1

            for neg in negatives:
                pf.write(json.dumps({**base, "positive": positive, "negative": neg}, ensure_ascii=False) + "\n")
                n_pairs += 1

    logger.info(
        "Derived %d triplets and %d pairs from %s → %s, %s",
        n_triplets, n_pairs, scored_path.name, triplet_path.name, pairs_path.name,
    )
# --- end inlined derive_triplets ---

logger = logging.getLogger(__name__)


# =============================================================================
# Inference-parity helpers
# =============================================================================


def format_query(query: str, persona_id: str | None) -> str:
    """Render a query exactly as ``Qwen3Embedder.encode_query`` does at inference.

    ``rag.embedder.Qwen3Embedder`` passes ``prompt=f"Instruct: {instruction}\\nQuery: "``
    to sentence-transformers, which concatenates it in front of the text. Reproducing
    that string here is what keeps the fine-tuned adapter usable by the serving path.
    Documents are deliberately *not* wrapped — inference encodes them bare.
    """
    if not persona_id:
        return query
    return f"Instruct: {render_profile(persona_id)}\nQuery: {query}"


def last_token_pool(
    last_hidden_states: torch.Tensor, attention_mask: torch.Tensor
) -> torch.Tensor:
    """Qwen3-Embedding's native pooling: take the final non-padding token.

    Padding-side agnostic, matching the reference implementation on the model card.
    sentence-transformers applies the same pooling at inference, so this is the
    single most important line for train/serve consistency.
    """
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    return last_hidden_states[
        torch.arange(last_hidden_states.shape[0], device=last_hidden_states.device),
        seq_lens,
    ]


def unwrap(model: nn.Module) -> nn.Module:
    """Return the underlying module whether or not *model* is DDP-wrapped."""
    return model.module if isinstance(model, DDP) else model


# =============================================================================
# Datasets — persona-conditioned, chunk_id-preserving
# =============================================================================


class TripletDataset(Dataset):
    """Reads {query, persona_id, positive, negatives: [...]} lines (hard_neg mode)."""

    def __init__(self, jsonl_path: str, max_negatives: int = 4) -> None:
        self.items: list[dict[str, Any]] = []
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line.strip())
                negs = obj.get("negatives", [])[:max_negatives]
                if not negs:
                    continue
                self.items.append(
                    {
                        "query": format_query(obj["query"], obj.get("persona_id")),
                        "positive": obj["positive"],
                        "negatives": negs,
                    }
                )

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        return self.items[idx]


class PairDataset(Dataset):
    """Reads {query, persona_id, positive, negative} lines (cartesian-expanded)."""

    def __init__(self, jsonl_path: str) -> None:
        self.items: list[dict[str, str]] = []
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line.strip())
                self.items.append(
                    {
                        "query": format_query(obj["query"], obj.get("persona_id")),
                        "positive": obj["positive"],
                        "negative": obj["negative"],
                    }
                )

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> dict[str, str]:
        return self.items[idx]


class ScoredDataset(Dataset):
    """Reads listwise scored groups for KL-distillation.

    Supports two JSONL formats:
      1. Old: {"query": "...", "document": "...", "score": 0.5}
      2. New: {"query": "...", "persona_id": "...",
               "docs": [{"chunk_id": "...", "text": "...", "teacher_score": 0.3}, ...]}

    Groups are keyed on ``(raw query, persona_id)``. Keying on the *rendered* query
    alone would merge the same question asked of different personas — the exact
    signal this stage is trying to learn. ``chunk_id`` and ``persona_id`` are kept
    on each item because corpus-level per-persona evaluation needs both.
    """

    def __init__(self, jsonl_path: str, max_documents: int = 20) -> None:
        grouped: dict[tuple[str, str], list[tuple[str, float, str]]] = {}
        order: list[tuple[str, str]] = []
        n_dup = 0
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line.strip())
                persona_id = obj.get("persona_id", "")
                key = (obj["query"], persona_id)
                if key not in grouped:
                    grouped[key] = []
                    order.append(key)
                else:
                    n_dup += 1

                if "docs" in obj:
                    for doc in obj["docs"]:
                        grouped[key].append(
                            (
                                doc["text"],
                                float(doc["teacher_score"]),
                                doc.get("chunk_id", ""),
                            )
                        )
                elif "document" in obj and "score" in obj:
                    grouped[key].append(
                        (obj["document"], float(obj["score"]), obj.get("chunk_id", ""))
                    )
                else:
                    continue  # unknown format

        if n_dup:
            logger.warning(
                "%s: %d duplicate (query, persona_id) rows merged into existing groups; "
                "their docs beyond max_documents=%d are dropped.",
                Path(jsonl_path).name,
                n_dup,
                max_documents,
            )

        self.items: list[dict[str, Any]] = []
        for key in order:
            triples = grouped[key][:max_documents]
            if len(triples) < 2:
                continue
            docs, scores, chunk_ids = zip(*triples, strict=True)
            raw_query, persona_id = key
            self.items.append(
                {
                    "query": format_query(raw_query, persona_id),
                    "raw_query": raw_query,
                    "persona_id": persona_id,
                    "documents": list(docs),
                    "scores": list(scores),
                    "chunk_ids": list(chunk_ids),
                }
            )

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        return self.items[idx]


# =============================================================================
# Model
# =============================================================================


class QwenEmbeddingModel(nn.Module):
    """Qwen3-Embedding wrapper: last-token pooling + L2 normalization, optional LoRA."""

    def __init__(
        self,
        model_name: str,
        use_gradient_checkpointing: bool = True,
        model_dtype: torch.dtype = torch.float16,
        lora_config: dict | None = None,
    ) -> None:
        super().__init__()
        # Eager attention materializes the full attention score matrix *and* upcasts it
        # for the softmax, costing rows x heads x width^2 x 4 bytes - 2.5 GB for a single
        # 10-sequence micro-batch at width 2048, allocated again during gradient
        # checkpointing's backward recompute. That is what OOM'd a 2xT4 run mid-epoch.
        # SDPA either drops the matrix entirely (mem-efficient kernel; flash needs sm_80,
        # so not on a T4) or falls back to math, which still materializes it but in fp16
        # with no fp32 softmax - half the memory. Both outcomes fit; there is no reason to
        # prefer eager here. No try/except: if this checkpoint cannot do SDPA we want the
        # ValueError now, not an OOM hours in with nothing in the log to explain it.
        self.model = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True,
            torch_dtype=model_dtype,
            attn_implementation="sdpa",
        )
        # Caching is incompatible with gradient checkpointing and useless for encoding.
        self.model.config.use_cache = False
        # Log what was actually installed - transformers can quietly resolve to something
        # other than the request, and the difference is a 2x swing in peak memory.
        logger.info(
            "attn_implementation=%s", getattr(self.model.config, "_attn_implementation", "?")
        )

        if lora_config is not None:
            from peft import LoraConfig, TaskType, get_peft_model

            # With a frozen base, the input to the first checkpointed block has
            # requires_grad=False, so recomputation yields no gradient at all.
            # This hook is what makes gradient checkpointing + PEFT actually train.
            if use_gradient_checkpointing:
                self.model.enable_input_require_grads()
            peft_cfg = LoraConfig(
                task_type=TaskType.FEATURE_EXTRACTION,
                inference_mode=False,
                r=lora_config.get("r", 8),
                lora_alpha=lora_config.get("alpha", 16),
                lora_dropout=lora_config.get("dropout", 0.1),
                target_modules=list(
                    lora_config.get("target_modules", ["q_proj", "k_proj", "v_proj", "o_proj"])
                ),
            )
            self.model = get_peft_model(self.model, peft_cfg)

        if use_gradient_checkpointing:
            # use_reentrant=False is required for DDP: the reentrant autograd path
            # hides parameter usage from DDP's bucketing and trips "marked ready twice".
            self.model.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        emb = last_token_pool(outputs.last_hidden_state, attention_mask)
        # fp32 before normalising: fp16 resolution near 1.0 is too coarse to rank
        # cosine similarities stably, and every downstream loss is a softmax over them.
        return F.normalize(emb.float(), p=2, dim=1)

    def save_adapter(self, path: str | Path) -> None:
        """Persist LoRA weights (or the full model when training without LoRA)."""
        self.model.save_pretrained(str(path))


def trim_pad(
    input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """Drop leading columns that are padding in *every* row of this micro-batch.

    Padding is left-side (``tokenizer.padding_side = "left"``), so real tokens are
    right-aligned and slicing ``[:, -keep:]`` can only remove pad. Nothing the model
    would have used is lost: pad positions are already excluded by the attention mask,
    and ``last_token_pool`` reads the final token, which left padding never touches.
    The absolute position of every real token shifts by the same amount, which is
    invisible to RoPE — it encodes *relative* position.
    """
    keep = int(attention_mask.sum(dim=1).max().item())
    return input_ids[:, -keep:], attention_mask[:, -keep:]


def encode_chunked(
    model: nn.Module,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    micro_batch: int,
) -> torch.Tensor:
    """Encode in length-sorted micro-batches and concatenate, keeping one autograd graph.

    The concatenated result is a single graph, so the listwise loss and its gradients
    are identical to one large forward. That also bounds what chunking buys on memory:
    because the graph is retained until ``backward()``, *stored* activations scale with
    total tokens per step regardless of *micro_batch* — ``batch_size`` is the knob for
    those. What *micro_batch* caps is the transient per-forward working set.

    It is also a padding lever, and that is where the real cost sits. ``collate_scored``
    pads to the longest sequence in the whole batch, but each micro-batch is its own
    forward and needs only its own width. Sorting by true length concentrates the few
    long documents into one micro-batch instead of letting each drag its neighbours up
    to 2048; ``trim_pad`` then slices off the columns that are pad for the entire chunk.
    On this corpus (median document 167 tokens, p95 2048) that removes ~65% of all
    tokens processed. Rows are restored to the caller's order before returning, which
    both call sites depend on — they ``view(B, D, -1)`` the result against
    ``gold_scores``.

    Consequence for tuning: a smaller *micro_batch* now buys real compute, not just a
    smaller transient buffer, traded against starving the GPU on tiny launches.
    """
    n = input_ids.size(0)
    if micro_batch <= 0 or n <= micro_batch:
        return model(*trim_pad(input_ids, attention_mask))

    # stable=True keeps the permutation reproducible under the run seed.
    order = torch.argsort(attention_mask.sum(dim=1), stable=True)
    parts = [
        model(*trim_pad(input_ids[sel], attention_mask[sel]))
        for sel in (order[i : i + micro_batch] for i in range(0, n, micro_batch))
    ]
    inverse = torch.empty_like(order)
    inverse[order] = torch.arange(n, device=order.device)
    return torch.cat(parts, dim=0)[inverse]


# =============================================================================
# Losses
# =============================================================================


def mnrl_loss(
    query_emb: torch.Tensor,
    pos_emb: torch.Tensor,
    neg_emb: torch.Tensor,
    temperature: float = 0.05,
) -> torch.Tensor:
    """Multiple Negatives Ranking Loss over one positive and N in-row negatives."""
    q = query_emb.float()
    pos_score = (q * pos_emb.float()).sum(dim=-1, keepdim=True)
    neg_scores = torch.bmm(neg_emb.float(), q.unsqueeze(-1)).squeeze(-1)
    all_scores = torch.cat([pos_score, neg_scores], dim=-1) / temperature
    labels = torch.zeros(q.size(0), dtype=torch.long, device=q.device)
    return F.cross_entropy(all_scores, labels)


def kd_loss(
    student_scores: torch.Tensor,
    gold_scores: torch.Tensor,
    student_temp: float = 0.05,
    teacher_temp: float = 0.2,
    doc_mask: torch.Tensor | None = None,
) -> torch.Tensor:
    """Listwise KL(teacher || student) over each group's candidate documents.

    Both temperatures matter more than they look. Student scores are cosine
    similarities in [-1, 1] and teacher scores lie in [0, 1]; at temperature 1.0 a
    softmax over 20 candidates is nearly uniform on *both* sides, leaving almost no
    gradient. *doc_mask* removes padded slots from the student's normalisation so
    they cannot absorb probability mass.
    """
    student = student_scores.float() / student_temp
    teacher = gold_scores.float() / teacher_temp
    if doc_mask is not None:
        student = student.masked_fill(~doc_mask, float("-inf"))
        teacher = teacher.masked_fill(~doc_mask, float("-inf"))
    student_logprobs = F.log_softmax(student, dim=-1)
    gold_probs = F.softmax(teacher, dim=-1)
    if doc_mask is not None:
        # log_softmax leaves -inf in the padded slots, and kl_div evaluates
        # target * (log target - input) there as 0 * inf = NaN. The teacher weight is
        # already 0 on pads, so any finite value works; zero keeps the term at exactly 0.
        student_logprobs = student_logprobs.masked_fill(~doc_mask, 0.0)
    return F.kl_div(student_logprobs, gold_probs, reduction="batchmean")


# =============================================================================
# Collators
# =============================================================================


def collate_triplets(
    batch: list[dict[str, Any]], tokenizer, max_length: int
) -> dict[str, torch.Tensor]:
    queries = [x["query"] for x in batch]
    positives = [x["positive"] for x in batch]
    max_negs = max(len(x["negatives"]) for x in batch)

    negs_flat: list[str] = []
    neg_mask: list[list[bool]] = []
    for x in batch:
        n = len(x["negatives"])
        negs_flat.extend(x["negatives"])
        negs_flat.extend([""] * (max_negs - n))
        neg_mask.append([True] * n + [False] * (max_negs - n))

    def tok(texts: list[str]):
        return tokenizer(
            texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

    q, p, n_tok = tok(queries), tok(positives), tok(negs_flat)
    B = len(batch)
    return {
        "q_ids": q["input_ids"],
        "q_mask": q["attention_mask"],
        "p_ids": p["input_ids"],
        "p_mask": p["attention_mask"],
        "n_ids": n_tok["input_ids"].view(B, max_negs, -1),
        "n_mask": n_tok["attention_mask"].view(B, max_negs, -1),
        "neg_mask": torch.tensor(neg_mask, dtype=torch.bool),
    }


def collate_pairs(
    batch: list[dict[str, str]], tokenizer, max_length: int
) -> dict[str, torch.Tensor]:
    triplet_batch = [
        {"query": x["query"], "positive": x["positive"], "negatives": [x["negative"]]}
        for x in batch
    ]
    return collate_triplets(triplet_batch, tokenizer, max_length)


def collate_scored(
    batch: list[dict[str, Any]], tokenizer, max_length: int
) -> dict[str, torch.Tensor]:
    queries = [x["query"] for x in batch]
    max_docs = max(len(x["documents"]) for x in batch)

    docs_flat: list[str] = []
    gold_scores: list[list[float]] = []
    doc_mask: list[list[bool]] = []
    for x in batch:
        n = len(x["documents"])
        docs_flat.extend(x["documents"])
        docs_flat.extend([""] * (max_docs - n))
        gold_scores.append(list(x["scores"]) + [0.0] * (max_docs - n))
        doc_mask.append([True] * n + [False] * (max_docs - n))

    def tok(texts: list[str]):
        return tokenizer(
            texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

    q, d = tok(queries), tok(docs_flat)
    B = len(batch)
    return {
        "q_ids": q["input_ids"],
        "q_mask": q["attention_mask"],
        "d_ids": d["input_ids"].view(B, max_docs, -1),
        "d_mask": d["attention_mask"].view(B, max_docs, -1),
        "gold_scores": torch.tensor(gold_scores, dtype=torch.float32),
        "doc_mask": torch.tensor(doc_mask, dtype=torch.bool),
    }


def _move(batch: dict[str, torch.Tensor], device: torch.device) -> dict[str, torch.Tensor]:
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


# =============================================================================
# Train / eval steps
# =============================================================================


def _mnrl_step(model, batch, temps, micro_batch):
    student_temp, _ = temps
    q_ids, q_mask = batch["q_ids"], batch["q_mask"]
    p_ids, p_mask = batch["p_ids"], batch["p_mask"]
    n_ids, n_mask = batch["n_ids"], batch["n_mask"]
    B, N, L = n_ids.shape
    q_emb = encode_chunked(model, q_ids, q_mask, micro_batch)
    p_emb = encode_chunked(model, p_ids, p_mask, micro_batch)
    n_emb = encode_chunked(model, n_ids.view(B * N, L), n_mask.view(B * N, L), micro_batch).view(
        B, N, -1
    )
    return mnrl_loss(q_emb, p_emb, n_emb, student_temp)


def _kd_step(model, batch, temps, micro_batch):
    student_temp, teacher_temp = temps
    q_ids, q_mask = batch["q_ids"], batch["q_mask"]
    d_ids, d_mask = batch["d_ids"], batch["d_mask"]
    B, D, L = d_ids.shape
    q_emb = encode_chunked(model, q_ids, q_mask, micro_batch)
    d_emb = encode_chunked(model, d_ids.view(B * D, L), d_mask.view(B * D, L), micro_batch).view(
        B, D, -1
    )
    student_scores = torch.einsum("bh,bdh->bd", q_emb, d_emb)
    return kd_loss(
        student_scores,
        batch["gold_scores"],
        student_temp,
        teacher_temp,
        batch["doc_mask"],
    )


def train_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    scaler,
    device,
    step_fn,
    temps,
    amp_dtype,
    use_amp,
    grad_clip,
    micro_batch,
    is_main: bool = True,
) -> float:
    model.train()
    total, n = 0.0, 0
    pbar = tqdm(loader, desc="train", leave=False, disable=not is_main)
    for batch in pbar:
        batch = _move(batch, device)
        with torch.autocast(device_type=device.type, enabled=use_amp, dtype=amp_dtype):
            loss = step_fn(model, batch, temps, micro_batch)

        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            # GradScaler starts at 65536, so the first fp16 backward usually overflows
            # and scaler.step() *skips* the update, halving the scale. Advancing the LR
            # schedule for an update that never happened is what triggers PyTorch's
            # "lr_scheduler.step() before optimizer.step()" warning - the call order
            # here is already the documented one; the skip is what it is complaining
            # about. A lowered scale is the signal that the step was dropped.
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            stepped = scaler.get_scale() >= scale_before
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            stepped = True
        if stepped:
            scheduler.step()

        total += loss.item()
        n += 1
        if is_main:
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total / max(n, 1)


@torch.no_grad()
def eval_epoch(model, loader, device, step_fn, temps, amp_dtype, use_amp, micro_batch) -> float:
    """Mean loss over the loader, all-reduced so every rank reports the *whole* val set.

    Without the reduction a DistributedSampler-sharded loader makes each rank report
    only its own slice, which then silently drives checkpoint selection.
    """
    model.eval()
    total, n = 0.0, 0
    for batch in tqdm(loader, desc="eval", leave=False, disable=True):
        batch = _move(batch, device)
        with torch.autocast(device_type=device.type, enabled=use_amp, dtype=amp_dtype):
            loss = step_fn(model, batch, temps, micro_batch)
        total += loss.item()
        n += 1

    if dist.is_available() and dist.is_initialized():
        stats = torch.tensor([total, float(n)], dtype=torch.float64, device=device)
        dist.all_reduce(stats, op=dist.ReduceOp.SUM)
        total, n = stats[0].item(), int(stats[1].item())
    return total / max(n, 1)


# =============================================================================
# Corpus-level evaluation (persona-aware)
# =============================================================================


def load_corpus(corpus_path: str) -> tuple[list[str], dict[str, int]]:
    """Load data/chunks/corpus.jsonl → (texts, chunk_id → row index)."""
    texts: list[str] = []
    chunk_id_to_row: dict[str, int] = {}
    with open(corpus_path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            chunk_id_to_row[obj["chunk_id"]] = len(texts)
            texts.append(obj["text"])
    return texts, chunk_id_to_row


@torch.no_grad()
def encode_texts(
    model,
    tokenizer,
    texts: list[str],
    device: torch.device,
    batch_size: int = 32,
    max_length: int = 512,
    amp_dtype: torch.dtype = torch.float16,
    use_amp: bool = True,
    desc: str | None = None,
) -> torch.Tensor:
    model.eval()
    out: list[torch.Tensor] = []
    rng = range(0, len(texts), batch_size)
    for i in tqdm(rng, desc=desc, leave=False, disable=desc is None):
        tokens = tokenizer(
            texts[i : i + batch_size],
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)
        with torch.autocast(device_type=device.type, enabled=use_amp, dtype=amp_dtype):
            emb = model(tokens["input_ids"], tokens["attention_mask"])
        out.append(emb.float())
    return torch.cat(out, dim=0)


@torch.no_grad()
def evaluate_retrieval(
    model,
    tokenizer,
    groups: list[dict[str, Any]],
    corpus_texts: list[str],
    chunk_id_to_row: dict[str, int],
    device: torch.device,
    top_k: int,
    relevance_top_m: int,
    batch_size: int = 32,
    max_length: int = 512,
    amp_dtype: torch.dtype = torch.float16,
    use_amp: bool = True,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """nDCG@1..K, Hit@1..K, Recall@K and MRR against the **full corpus**.

    Returns ``(metrics, per_query)``: aggregates per persona and overall, plus the
    raw per-group vectors behind them. The vectors are what make significance testing
    possible — a mean can yield a marginal confidence interval but never a *paired*
    one, and the paired test is the powerful one here because every epoch scores the
    same queries. They are aligned by position, which is sound because group order and
    the skip decision below depend only on the data, never on the model.

    Ranking a group's own 20 candidates measures reranking accuracy, not retrieval;
    the real question is whether the relevant chunks surface out of the whole index.

    Two relevance notions are reported deliberately:

    * **nDCG** (primary) grades by raw ``teacher_score``. Gain is linear, not
      ``2^rel - 1``: scores already live in [0, 1], so exponential gain only compresses
      them. Grading matters because the teacher's scores are far from flat — mean score
      by rank runs 0.757 / 0.569 / 0.447 / 0.372 / 0.323 — and because any binary cutoff
      lands on a near-tie: the rank-3 to rank-4 gap is under 0.05 in 53% of val groups.
      nDCG also has ceiling 1.0 at *every* k, so nDCG@1 is directly readable.
    * **Recall/Hit/MRR** keep the binary top-*relevance_top_m* set. Retained on purpose:
      nDCG is shaped like the KD objective (both range over the teacher's graded
      distribution), so a coarser, differently-shaped metric belongs beside it.

    Note Recall@k divides by the size of that set, so Recall@1 could never exceed
    1/*relevance_top_m*; it is reported at k=K only, where the ceiling is 1.0.

    Only the group's own judged chunks have gains — the other ~150 corpus chunks score
    0 even if genuinely relevant. That incomplete-judgments bias predates nDCG and
    applies to Recall/MRR identically; see docs/methodology.md.

    The corpus is embedded once and reused across groups — re-encoding each group's
    docs made validation roughly 15x slower for identical numbers.
    """
    corpus_matrix = encode_texts(
        model,
        tokenizer,
        corpus_texts,
        device,
        batch_size,
        max_length,
        amp_dtype,
        use_amp,
        desc="eval: corpus",
    )

    query_matrix = encode_texts(
        model,
        tokenizer,
        [g["query"] for g in groups],
        device,
        batch_size,
        max_length,
        amp_dtype,
        use_amp,
    )
    sims_all = query_matrix @ corpus_matrix.T  # (G, N)

    # Position i (0-based) in a ranking carries discount 1/log2(i+2).
    discount = 1.0 / np.log2(np.arange(2, top_k + 2))
    ks = list(range(1, top_k + 1))
    names = [f"ndcg@{k}" for k in ks] + [f"hit@{k}" for k in ks] + [f"recall@{top_k}", "mrr"]

    values: dict[str, list[float]] = {name: [] for name in names}
    personas: list[str] = []
    n_skipped = 0

    for gi, group in enumerate(groups):
        scored = sorted(
            zip(group["chunk_ids"], group["scores"], strict=True),
            key=lambda t: t[1],
            reverse=True,
        )
        # Gains only for judged chunks that exist in the corpus. Restricting the ideal
        # ranking the same way keeps nDCG's ceiling attainable — grading against chunks
        # the retriever cannot return would depress every score by a constant.
        gains_by_row = {
            chunk_id_to_row[cid]: float(s) for cid, s in scored if cid in chunk_id_to_row
        }
        relevant_rows = {
            chunk_id_to_row[cid] for cid, _ in scored[:relevance_top_m] if cid in chunk_id_to_row
        }
        if not relevant_rows:
            n_skipped += 1
            continue

        ranked = torch.argsort(sims_all[gi], descending=True).tolist()
        head = ranked[:top_k]

        gains = np.array([gains_by_row.get(row, 0.0) for row in head])
        ideal = np.zeros(top_k)
        best_gains = sorted(gains_by_row.values(), reverse=True)[:top_k]
        ideal[: len(best_gains)] = best_gains
        dcg = np.cumsum(gains * discount)
        idcg = np.cumsum(ideal * discount)
        ndcg = np.divide(dcg, idcg, out=np.zeros_like(dcg), where=idcg > 0)

        found = np.cumsum([1.0 if row in relevant_rows else 0.0 for row in head])

        rr = 0.0
        for rank, row in enumerate(ranked, start=1):
            if row in relevant_rows:
                rr = 1.0 / rank
                break

        for i, k in enumerate(ks):
            values[f"ndcg@{k}"].append(float(ndcg[i]))
            values[f"hit@{k}"].append(1.0 if found[i] > 0 else 0.0)
        values[f"recall@{top_k}"].append(float(found[-1]) / len(relevant_rows))
        values["mrr"].append(rr)
        personas.append(group.get("persona_id") or "unknown")

    if n_skipped:
        logger.warning(
            "%d/%d val groups had no chunk_id matching the corpus and were skipped.",
            n_skipped,
            len(groups),
        )

    persona_arr = np.array(personas)
    metrics: dict[str, Any] = {}
    for name in names:
        arr = np.array(values[name])
        metrics[name] = {"overall": float(arr.mean()) if arr.size else 0.0}
        for persona_id in sorted(set(personas)):
            sel = arr[persona_arr == persona_id]
            metrics[name][persona_id] = float(sel.mean()) if sel.size else 0.0

    # Rounded: this lands in training_log.json once per epoch, and 4 dp is well past
    # the resolution of a 276-query mean.
    per_query = {"persona_ids": personas} | {
        name: [round(v, 4) for v in values[name]] for name in names
    }
    return metrics, per_query


def is_better(candidate: dict, current_best: dict | None, top_k: int) -> bool:
    """Best = highest overall Recall@K; ties → lower val KD loss; ties → higher MRR.

    Selecting on KD loss alone would reward an encoder that matches the teacher's
    distribution while retrieving worse — the reward-hacking guard from the methodology.
    """
    if current_best is None:
        return True
    key = f"recall@{top_k}"
    if candidate[key]["overall"] != current_best[key]["overall"]:
        return candidate[key]["overall"] > current_best[key]["overall"]
    if candidate["val_loss"] != current_best["val_loss"]:
        return candidate["val_loss"] < current_best["val_loss"]
    return candidate["mrr"]["overall"] > current_best["mrr"]["overall"]


def validate(
    model,
    val_loader,
    device: torch.device,
    step_fn,
    temps,
    amp_dtype,
    use_amp,
    micro_batch,
    *,
    tokenizer,
    eval_groups,
    corpus_texts,
    chunk_id_to_row,
    top_k: int,
    relevance_top_m: int,
    eval_batch_size: int,
    max_length: int,
    can_eval_retrieval: bool,
    is_main: bool,
) -> tuple[float, dict | None, dict | None]:
    """Val KD loss on **every** rank; retrieval metrics on rank 0 only.

    Every rank must call this: ``eval_epoch`` all-reduces, so a rank that skips it
    leaves the others blocked on a collective forever — a hang with no traceback.
    Retrieval metrics are rank-0 only and the caller must ``dist.barrier()`` after,
    so other ranks cannot race ahead into the next epoch mid-save.
    """
    val_loss = eval_epoch(
        model, val_loader, device, step_fn, temps, amp_dtype, use_amp, micro_batch
    )
    if not (is_main and can_eval_retrieval):
        return val_loss, None, None

    metrics, per_query = evaluate_retrieval(
        unwrap(model),
        tokenizer,
        eval_groups,
        corpus_texts,
        chunk_id_to_row,
        device,
        top_k,
        relevance_top_m,
        eval_batch_size,
        max_length,
        amp_dtype,
        use_amp,
    )
    metrics["val_loss"] = val_loss
    return val_loss, metrics, per_query


def log_eval_block(header: str, metrics: dict, top_k: int, suffix: str = "") -> None:
    """Curves first, then the headline scalars, then one line per persona."""
    ks = range(1, top_k + 1)
    logger.info("%s%s", header, suffix)
    logger.info(
        "  nDCG@1..%d   %s", top_k, " ".join(f"{metrics[f'ndcg@{k}']['overall']:.3f}" for k in ks)
    )
    logger.info(
        "  Hit@1..%d    %s", top_k, " ".join(f"{metrics[f'hit@{k}']['overall']:.3f}" for k in ks)
    )
    logger.info(
        "  Recall@%d %.4f | MRR %.4f",
        top_k,
        metrics[f"recall@{top_k}"]["overall"],
        metrics["mrr"]["overall"],
    )
    for persona_id in sorted(k for k in metrics["mrr"] if k != "overall"):
        logger.info(
            "    %-9s nDCG@%d %.4f | Hit@%d %.4f | Recall@%d %.4f | MRR %.4f",
            persona_id,
            top_k,
            metrics[f"ndcg@{top_k}"][persona_id],
            top_k,
            metrics[f"hit@{top_k}"][persona_id],
            top_k,
            metrics[f"recall@{top_k}"][persona_id],
            metrics["mrr"][persona_id],
        )


# =============================================================================
# Training entry point
# =============================================================================


def train(config: dict, rank: int = 0, world_size: int = 1, local_rank: int = 0) -> None:
    """Main training routine. Runs single-GPU when *world_size* is 1, DDP otherwise."""
    is_ddp = world_size > 1
    is_main = rank == 0

    seed = config.get("seed", 42)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    train_cfg = config["training"]
    emb_cfg = config["embedder"]
    eval_cfg = config["eval"]

    if torch.cuda.is_available() and train_cfg.get("device", "cuda") == "cuda":
        device = torch.device(f"cuda:{local_rank}")
        torch.cuda.set_device(device)
    else:
        device = torch.device("cpu")

    precision = train_cfg.get("precision", "fp32")
    use_amp = device.type == "cuda" and precision in ("fp16", "bf16")
    # bf16 Tensor Cores need Ampere+ (compute capability >= 8). Kaggle's T4 is Turing,
    # so honour the configured precision instead of hardcoding bf16 as before.
    bf16_ok = device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8
    if precision == "bf16" and not bf16_ok:
        logger.warning("bf16 requested but unsupported on this GPU; falling back to fp16.")
        precision = "fp16"
    amp_dtype = torch.bfloat16 if precision == "bf16" else torch.float16
    model_dtype = (
        torch.float32 if device.type == "cpu" else (torch.bfloat16 if bf16_ok else torch.float16)
    )
    # fp16 gradients underflow without loss scaling; bf16 has the range to skip it.
    scaler = torch.amp.GradScaler("cuda") if (use_amp and amp_dtype is torch.float16) else None

    mode = config.get("mode", "reader_kd")
    fmt = config.get("format", "triplets")
    train_data = Path(config["data"]["train_data"])
    corpus_path = config["data"].get("corpus_path", "data/chunks/corpus.jsonl")
    output_dir = Path(config.get("checkpoint_dir", "./ropg_kd_checkpoints"))

    if mode == "hard_neg":
        suffix = "triplets" if fmt == "triplets" else "pairs"
        train_path = train_data / f"train_{suffix}.jsonl"
        val_path = train_data / f"val_{suffix}.jsonl"
    else:
        train_path = train_data / "train.jsonl"
        val_path = train_data / "val.jsonl"

    if mode == "hard_neg" and not train_path.exists():
        scored_train = train_data / "train.jsonl"
        if not scored_train.exists():
            raise FileNotFoundError(
                f"Neither {train_path} nor {scored_train} found. "
                "Run data generation first (src/data/gen_ropg_data.py)."
            )
        if is_main:
            logger.warning("%s missing — deriving from scored data.", train_path.name)
            for sp in (train_data / "train.jsonl", train_data / "val.jsonl"):
                if sp.exists():
                    derive_triplets(sp, train_data, train_cfg.get("max_negatives", 4))
        if is_ddp:
            dist.barrier()

    batch_size = train_cfg["batch_size"]
    num_epochs = train_cfg["epochs"]
    grad_clip = train_cfg.get("grad_clip", 1.0)
    micro_batch = train_cfg.get("doc_micro_batch", 2)
    max_negatives = train_cfg.get("max_negatives", 4)
    max_documents = train_cfg.get("max_documents", 20)
    temps = (
        train_cfg.get("student_temp", 0.05),
        train_cfg.get("teacher_temp", 0.2),
    )
    model_name = emb_cfg["model"]
    max_length = emb_cfg.get("max_seq_length", 2048)
    top_k = eval_cfg.get("top_k", 5)
    relevance_top_m = eval_cfg.get("relevance_top_m", 3)
    eval_batch_size = eval_cfg.get("eval_batch_size", 8)

    if is_main:
        logger.info(
            "Device: %s | world_size: %d | precision: %s | amp: %s | scaler: %s",
            device,
            world_size,
            precision,
            use_amp,
            scaler is not None,
        )

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    # Qwen3-Embedding is a decoder; left padding keeps the final real token last,
    # which is what last_token_pool reads.
    tokenizer.padding_side = "left"

    if mode == "hard_neg":
        if fmt == "triplets":
            train_ds = TripletDataset(str(train_path), max_negatives)
            val_ds = TripletDataset(str(val_path), max_negatives) if val_path.exists() else None
            collate_fn = lambda b: collate_triplets(b, tokenizer, max_length)  # noqa: E731
        else:
            train_ds = PairDataset(str(train_path))
            val_ds = PairDataset(str(val_path)) if val_path.exists() else None
            collate_fn = lambda b: collate_pairs(b, tokenizer, max_length)  # noqa: E731
        step_fn = _mnrl_step
    else:
        train_ds = ScoredDataset(str(train_path), max_documents)
        val_ds = ScoredDataset(str(val_path), max_documents) if val_path.exists() else None
        collate_fn = lambda b: collate_scored(b, tokenizer, max_length)  # noqa: E731
        step_fn = _kd_step

    # Corpus-level retrieval metrics need chunk_ids, which only the scored format
    # carries. In hard_neg mode the scored val file supplies them.
    scored_val_path = train_data / "val.jsonl"
    eval_groups = (
        ScoredDataset(str(scored_val_path), max_documents).items
        if scored_val_path.exists()
        else []
    )
    corpus_texts, chunk_id_to_row = (
        load_corpus(corpus_path) if Path(corpus_path).exists() else ([], {})
    )
    can_eval_retrieval = bool(eval_groups and corpus_texts)
    if is_main and not can_eval_retrieval:
        logger.warning(
            "Corpus-level retrieval eval disabled (corpus=%s, val groups=%d); "
            "checkpoint selection falls back to val loss.",
            corpus_path,
            len(eval_groups),
        )

    if is_main:
        logger.info("Train samples: %d", len(train_ds))
        logger.info("Val samples  : %d", len(val_ds) if val_ds else 0)
        logger.info("Corpus chunks: %d", len(corpus_texts))

    train_sampler = (
        DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True)
        if is_ddp
        else None
    )
    val_sampler = (
        DistributedSampler(val_ds, num_replicas=world_size, rank=rank, shuffle=False)
        if (is_ddp and val_ds)
        else None
    )
    num_workers = train_cfg.get("num_workers", 2)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=(train_sampler is None),
        sampler=train_sampler,
        num_workers=num_workers,
        collate_fn=collate_fn,
        drop_last=True,
    )
    val_loader = (
        DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            sampler=val_sampler,
            num_workers=num_workers,
            collate_fn=collate_fn,
            drop_last=False,
        )
        if val_ds
        else None
    )

    model = QwenEmbeddingModel(
        model_name,
        use_gradient_checkpointing=train_cfg.get("gradient_checkpointing", True),
        model_dtype=model_dtype,
        lora_config=config.get("lora"),
    ).to(device)
    if is_main and config.get("lora"):
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        logger.info(
            "Trainable params: %d / %d (%.3f%%)", trainable, total, 100 * trainable / total
        )

    if is_ddp:
        # static_graph=True lets the same parameters be used by several forwards per
        # step (doc micro-batching) without DDP raising "marked ready twice", and is
        # also what makes non-reentrant gradient checkpointing safe under DDP.
        model = DDP(model, device_ids=[local_rank], output_device=local_rank, static_graph=True)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=train_cfg["lr"],
        weight_decay=train_cfg.get("weight_decay", 0.01),
    )
    steps_per_epoch = len(train_loader)
    total_steps = max(1, steps_per_epoch * num_epochs)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * train_cfg.get("warmup_ratio", 0.1)), total_steps
    )
    if is_main:
        logger.info("Steps/epoch: %d | total: %d", steps_per_epoch, total_steps)

    output_dir.mkdir(parents=True, exist_ok=True)
    save_dir = output_dir / "checkpoint-best"
    best: dict | None = None
    best_epoch = 0
    epoch_metrics: list[dict[str, Any]] = []
    baseline: dict[str, Any] | None = None

    # Epoch 0: the untrained model. PEFT zero-initializes lora_B and eval() disables
    # dropout, so the adapter is exactly the identity here - these numbers *are* the
    # frozen Qwen3-Embedding baseline (Rung 1), measured through the same val set,
    # relevance definition and code path as every trained epoch. It also exercises the
    # whole eval path (corpus encoding, all-reduce, barrier) in minutes rather than
    # after a two-hour training epoch. Deliberately not eligible for checkpoint-best:
    # an identity adapter winning would silently make Rung 3 equal Rung 1.
    if val_loader:
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)
        base_loss, base_metrics, base_per_query = validate(
            model,
            val_loader,
            device,
            step_fn,
            temps,
            amp_dtype,
            use_amp,
            micro_batch,
            tokenizer=tokenizer,
            eval_groups=eval_groups,
            corpus_texts=corpus_texts,
            chunk_id_to_row=chunk_id_to_row,
            top_k=top_k,
            relevance_top_m=relevance_top_m,
            eval_batch_size=eval_batch_size,
            max_length=max_length,
            can_eval_retrieval=can_eval_retrieval,
            is_main=is_main,
        )
        if is_main:
            # No train_loss key: its absence is how the results cell recognises epoch 0.
            baseline = {"epoch": 0, "val_loss": base_loss}
            if base_metrics is not None:
                baseline.update({k: v for k, v in base_metrics.items() if k != "val_loss"})
                baseline["per_query"] = base_per_query
                log_eval_block(
                    f"Baseline (untrained, epoch 0) | val {base_loss:.4f}", base_metrics, top_k
                )
            else:
                logger.info("Baseline (untrained, epoch 0) | val %.4f", base_loss)
            if device.type == "cuda":
                logger.info(
                    "Epoch 0 peak GPU: allocated %.2f GiB (eval only)",
                    torch.cuda.max_memory_allocated(device) / 2**30,
                )
            epoch_metrics.append(baseline)
        if is_ddp:
            dist.barrier()

    for epoch in range(1, num_epochs + 1):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)

        train_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            scaler,
            device,
            step_fn,
            temps,
            amp_dtype,
            use_amp,
            grad_clip,
            micro_batch,
            is_main,
        )
        entry: dict[str, Any] = {"epoch": epoch, "train_loss": train_loss}

        if val_loader:
            val_loss, metrics, per_query = validate(
                model,
                val_loader,
                device,
                step_fn,
                temps,
                amp_dtype,
                use_amp,
                micro_batch,
                tokenizer=tokenizer,
                eval_groups=eval_groups,
                corpus_texts=corpus_texts,
                chunk_id_to_row=chunk_id_to_row,
                top_k=top_k,
                relevance_top_m=relevance_top_m,
                eval_batch_size=eval_batch_size,
                max_length=max_length,
                can_eval_retrieval=can_eval_retrieval,
                is_main=is_main,
            )
            entry["val_loss"] = val_loss

            # Checkpointing happens on rank 0 only; the barrier below stops other ranks
            # racing into the next epoch mid-save.
            if is_main:
                if metrics is not None:
                    entry.update({k: v for k, v in metrics.items() if k != "val_loss"})
                    entry["per_query"] = per_query
                    if is_better(metrics, best, top_k):
                        best, best_epoch = metrics, epoch
                        unwrap(model).save_adapter(save_dir)
                        tokenizer.save_pretrained(save_dir)
                    log_eval_block(
                        f"Epoch {epoch}/{num_epochs} | train {train_loss:.4f} | "
                        f"val {val_loss:.4f}",
                        metrics,
                        top_k,
                        "  *best*" if best_epoch == epoch else "",
                    )
                else:
                    if best is None or val_loss < best["val_loss"]:
                        best, best_epoch = {"val_loss": val_loss}, epoch
                        unwrap(model).save_adapter(save_dir)
                        tokenizer.save_pretrained(save_dir)
                    logger.info(
                        "Epoch %d/%d | train %.4f | val %.4f%s",
                        epoch,
                        num_epochs,
                        train_loss,
                        val_loss,
                        "  *best (loss)*" if best_epoch == epoch else "",
                    )
            if is_ddp:
                dist.barrier()
        elif is_main:
            logger.info("Epoch %d/%d | train %.4f", epoch, num_epochs, train_loss)

        if device.type == "cuda":
            # Logged on *every* rank, not just rank 0: peak memory is data-dependent
            # (the attention buffer scales with rows x width^2, and the sampler reshuffles
            # each epoch), so the rank that OOMs is not necessarily the one that reports.
            # A flat series across epochs means the peak is a property of the worst batch;
            # a climbing one means something is actually being retained.
            entry["peak_gib"] = torch.cuda.max_memory_allocated(device) / 2**30
            logger.info(
                "Epoch %d peak GPU: allocated %.2f GiB | reserved %.2f GiB",
                epoch,
                entry["peak_gib"],
                torch.cuda.max_memory_reserved(device) / 2**30,
            )

        epoch_metrics.append(entry)
        if device.type == "cuda":
            torch.cuda.empty_cache()

    if is_main and baseline is not None and best is not None:
        # The question the baseline exists to answer, stated once at the end. A run that
        # never beats the untrained encoder is a real result and belongs in the log at
        # WARNING, not buried in a table of per-epoch numbers.
        improved = False
        for name in (f"ndcg@{top_k}", f"recall@{top_k}", "mrr"):
            if name not in baseline or name not in best:
                continue
            before, after = baseline[name]["overall"], best[name]["overall"]
            logger.info(
                "%-9s baseline %.4f -> best (epoch %d) %.4f   %+.4f",
                name,
                before,
                best_epoch,
                after,
                after - before,
            )
            improved |= after > before
        if not improved:
            logger.warning(
                "No epoch beat the untrained baseline on any headline metric. "
                "checkpoint-best is still a trained adapter, not the baseline - treat "
                "this as a negative result, not a checkpoint to ship."
            )

    if is_main:
        final_dir = output_dir / "checkpoint-final"
        unwrap(model).save_adapter(final_dir)
        tokenizer.save_pretrained(final_dir)
        (output_dir / "training_log.json").write_text(
            json.dumps(
                {
                    "config": config,
                    "seed": seed,
                    "world_size": world_size,
                    "epoch_metrics": epoch_metrics,
                    "best_epoch": best_epoch,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        logger.info("Best epoch %d → %s | final → %s", best_epoch, save_dir, final_dir)


# =============================================================================
# Launchers
# =============================================================================


def main_worker(config: dict) -> None:
    """Single training process. Launched once per GPU by ``torchrun``.

    Rank information comes from the environment, which is torchrun's contract and
    also degrades correctly to one process under a bare ``python -m rl.ropg_kd``.
    """
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    rank = int(os.environ.get("RANK", local_rank))
    world_size = int(os.environ.get("WORLD_SIZE", 1))

    logging.basicConfig(
        level=logging.INFO,
        format=f"%(levelname)s [rank{rank}] %(message)s",
        force=True,
    )
    if world_size > 1:
        # torchrun sets these; the defaults only matter under a manual launch.
        os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
        os.environ.setdefault("MASTER_PORT", "29500")
        torch.cuda.set_device(local_rank)
        dist.init_process_group(
            backend="nccl",
            init_method="env://",
            rank=rank,
            world_size=world_size,
            # NCCL's default is ~30 minutes, long enough that a rendezvous stall
            # reads as an infinite hang. Fail loudly instead: a healthy two-T4
            # handshake takes seconds.
            timeout=timedelta(minutes=10),
        )
    try:
        train(config, rank=rank, world_size=world_size, local_rank=local_rank)
    finally:
        if world_size > 1 and dist.is_initialized():
            dist.destroy_process_group()


def main() -> None:
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    parser = argparse.ArgumentParser(
        description="ROPG-KD: fine-tune the Qwen3-Embedding encoder with KD."
    )
    parser.add_argument("--config", required=True, help="Path to train_ropg.yaml.")
    parser.add_argument("--train_data", default=None, help="Override data.train_data.")
    parser.add_argument("--corpus_path", default=None, help="Override data.corpus_path.")
    parser.add_argument("--checkpoint_dir", default=None, help="Override checkpoint_dir.")
    args = parser.parse_args()

    with open(args.config, encoding="utf-8") as f:
        config = yaml.safe_load(f)
    if args.train_data:
        config["data"]["train_data"] = args.train_data
    if args.corpus_path:
        config["data"]["corpus_path"] = args.corpus_path
    if args.checkpoint_dir:
        config["checkpoint_dir"] = args.checkpoint_dir

    # One rank per process. Under torchrun this file is executed once per GPU and
    # main_worker picks its rank up from the environment; run bare, it is a single
    # process with world_size 1.
    main_worker(config)


if __name__ == "__main__":
    main()


## Launch

`torchrun` starts one process per GPU; Jupyter shows their output as it arrives, and
`--tee` keeps a per-rank copy under `torchrun_logs/`. The health check runs on rank 0
inside the worker.

In [ ]:
import torch

n_gpus = torch.cuda.device_count()
print(f"Visible GPUs: {n_gpus}")
for i in range(n_gpus):
    print(f"  cuda:{i} {torch.cuda.get_device_name(i)}")

# torchrun children are independent processes, so nothing this notebook does to the
# GPU can starve them - the old "never touch CUDA in the parent" rule was a spawn
# constraint and no longer applies.


In [ ]:
# torchrun starts one process per GPU. The NCCL settings matter on Kaggle: the
# container gives NCCL a small /dev/shm and the two T4s have no NVLink, so its default
# transports can stall at init. The fallback path is slower, but LoRA gradients are
# ~5M params so the cost is irrelevant here.
!NCCL_SHM_DISABLE=1 NCCL_P2P_DISABLE=1 NCCL_DEBUG=WARN PYTHONUNBUFFERED=1 \
    torchrun --standalone --nproc_per_node={n_gpus} \
    --tee 3 --log-dir torchrun_logs \
    ropg_kd_worker.py --config {CFG_PATH} \
    ; echo "[torchrun exit $?]"


### If the run fails

- **Non-zero exit.** The cell ends with `[torchrun exit N]` — worth checking, since a
  `!` command does not otherwise fail the cell. The traceback is in the output above;
  per-rank copies are under `torchrun_logs/`, which is where a rank-1-only failure
  shows up.
- **Stalls at startup.** `init_process_group` is capped at 10 minutes, so a rendezvous
  problem now raises instead of hanging. Set `NCCL_DEBUG=INFO` to see the handshake.
- **Isolating DDP.** Run `--nproc_per_node=1` — that path never calls
  `init_process_group`, so if it also fails the problem is in the training code.
- **Out of memory - read the allocation size first.** Peak memory here is *not* stable
  step to step, so an OOM can land many epochs in and still not be a leak. The dominant
  transient is the attention score matrix at `rows x heads x width^2 x dtype`, where
  `rows` is `doc_micro_batch` and `width` is set by the longest document the shuffle put
  in that batch (2048 for ~10 of the corpus's 171 chunks, ~167 for the median one). Check
  the `Epoch N peak GPU` lines: flat across epochs means the peak belongs to the worst
  batch, climbing means something is genuinely being retained.
  1. Confirm the startup log says `attn_implementation=sdpa`. Under `eager` this buffer
     is materialized *and* upcast to fp32 - 2.5 GB for one 10-row micro-batch at 2048.
  2. Halve `doc_micro_batch` (10 -> 5). Halves that buffer directly. Costs throughput:
     small launches starve the T4, which is what made `doc_micro_batch: 2` so slow.
  3. Lower `training.batch_size`. The knob for *retained* activations - all micro-forwards
     stay in one autograd graph until `backward()` - but it also changes the effective
     batch (`batch_size x world_size`), so the loss curve stops being comparable.
- **Out of memory during `eval: corpus`.** Different site, same cause: `encode_texts` is
  unsorted, so one long chunk widens a whole `eval.eval_batch_size` batch. It is `no_grad`
  and rank-0-only, ~5% of an epoch, so drop it to 8 without thinking twice.
- **Never turn off `gradient_checkpointing` on a T4.** It is not a speed knob to trade
  away: one `batch_size: 1` step retains ~36 GB of activations without it against ~2.5 GB
  with it, so nothing fits in 16 GB. It costs one extra forward (~33%), while the sorted
  micro-batching in `encode_chunked` is worth ~2.2x - fix throughput there instead.
- **`ImportError: incompatible version of torchao`** during `get_peft_model`. The setup
  cell should have removed torchao; if the image reinstalls it, re-run that cell. No
  kernel restart is needed - torchrun starts fresh interpreters that re-import
  everything.

## Results

Epoch 0 is the untrained model: a fresh LoRA adapter is the identity function, so those
numbers are the frozen Qwen3-Embedding baseline (Rung 1) measured through this exact
pipeline. Everything after is compared against it.

In [ ]:
import json
from pathlib import Path

log = Path(CFG["checkpoint_dir"]) / "training_log.json"
if log.exists():
    data = json.loads(log.read_text(encoding="utf-8"))
    print(f"Best epoch: {data['best_epoch']}  (world_size={data.get('world_size', 1)})")
    top_k = CFG["eval"]["top_k"]
    ks = range(1, top_k + 1)
    for e in data["epoch_metrics"]:
        # Epoch 0 has no train_loss - it is a validation pass, not a trained epoch.
        head = "baseline (untrained)" if e["epoch"] == 0 else f"epoch {e['epoch']}"
        line = f"  {head}:"
        if "train_loss" in e:
            line += f" train {e['train_loss']:.4f} |"
        line += f" val {e['val_loss']:.4f}"
        print(line)
        if f"ndcg@{top_k}" in e:
            print("      nDCG@1..%d  %s" % (top_k, " ".join(f"{e[f'ndcg@{k}']['overall']:.3f}" for k in ks)))
            print("      Hit@1..%d   %s" % (top_k, " ".join(f"{e[f'hit@{k}']['overall']:.3f}" for k in ks)))
            rec = e[f"recall@{top_k}"]
            print(f"      Recall@{top_k} {rec['overall']:.4f} | MRR {e['mrr']['overall']:.4f}")
            for persona in sorted(k for k in rec if k != "overall"):
                print(f"        {persona:<9} nDCG@{top_k} {e[f'ndcg@{top_k}'][persona]:.4f}"
                      f" | Recall@{top_k} {rec[persona]:.4f} | MRR {e['mrr'][persona]:.4f}")
else:
    print(f"No training log at {log}")


## Significance vs. the baseline

Each epoch scores the *same* val queries as epoch 0, so the comparison is **paired** and
must be tested that way. Marginal confidence intervals on two means overlap far more often
than the paired difference contains zero — with ~276 queries a single epoch's interval is
roughly ±0.04 wide, enough to hide a real gain of that size. Resampling the queries and
taking the difference each time cancels the shared query difficulty, which is what makes
the test sharp enough to be worth running.

In [ ]:
import numpy as np

if log.exists() and any("per_query" in e for e in data["epoch_metrics"]):
    rng = np.random.default_rng(CFG["seed"])
    B = 5000
    epochs = {e["epoch"]: e["per_query"] for e in data["epoch_metrics"] if "per_query" in e}
    base = epochs.get(0)
    if base is None:
        print("No epoch-0 baseline in this log - nothing to compare against.")
    else:
        # Pairing is by position: group order and the skip decision depend only on the
        # data, never on the model, so index i is the same query in every epoch.
        n = len(base["mrr"])
        idx = rng.integers(0, n, (B, n))
        names = [f"ndcg@{top_k}", f"hit@{top_k}", f"recall@{top_k}", "mrr"]
        for ep in sorted(k for k in epochs if k != 0):
            print(f"epoch {ep} vs baseline   (n={n} queries, {B} bootstrap resamples)")
            for name in names:
                a, b = np.array(base[name]), np.array(epochs[ep][name])
                if a.shape != b.shape:
                    print(f"  {name:<10} skipped - length mismatch, pairing is unsafe")
                    continue
                d = b - a
                lo, hi = np.percentile(d[idx].mean(1), [2.5, 97.5])
                flag = "" if (lo > 0 or hi < 0) else "   (CI includes 0)"
                print(f"  {name:<10} {a.mean():.4f} -> {b.mean():.4f}"
                      f"   {d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]{flag}")
else:
    print("No per-query vectors in the log - re-run training to enable paired testing.")


## Config drift check

The inline `CFG` above duplicates `configs/train_ropg.yaml`. When the repo is
reachable, compare them — this is the check that would have caught the earlier
`lr`, `max_seq_length` and `best_metric` drift.

In [ ]:
import yaml
from pathlib import Path

_candidates = [Path("configs/train_ropg.yaml"), Path("../configs/train_ropg.yaml")]
_cfg_path = next((p for p in _candidates if p.exists()), None)

if _cfg_path is None:
    print("configs/train_ropg.yaml not reachable — skipping drift check.")
else:
    ref = yaml.safe_load(_cfg_path.read_text(encoding="utf-8"))
    # Paths and checkpoint_dir are runtime-specific and expected to differ.
    SKIP = {("data", "train_data"), ("data", "corpus_path"), ("checkpoint_dir",)}

    def walk(a, b, path=()):
        if path in SKIP:
            return
        if isinstance(a, dict) and isinstance(b, dict):
            for k in sorted(set(a) | set(b)):
                if k not in a:
                    print(f"  MISSING in notebook CFG: {'.'.join(path + (k,))} = {b[k]!r}")
                elif k not in b:
                    print(f"  EXTRA in notebook CFG:   {'.'.join(path + (k,))} = {a[k]!r}")
                else:
                    walk(a[k], b[k], path + (k,))
        elif a != b:
            print(f"  DRIFT {'.'.join(path)}: notebook={a!r} yaml={b!r}")

    print(f"Comparing CFG against {_cfg_path}:")
    walk(CFG, ref)
    print("Done. No lines above means the two agree.")


## Download

In [ ]:
import subprocess
from pathlib import Path

if RUNTIME != "colab":
    out = Path(CFG["checkpoint_dir"])
    subprocess.run(
        ["zip", "-r", "ropg_kd_model.zip", out.name + "/checkpoint-best", out.name + "/training_log.json"],
        cwd=str(out.parent),
        check=True,
    )
    print(f"{out.parent / 'ropg_kd_model.zip'} created.")
else:
    print("Colab: checkpoints already on Google Drive - skipping zip.")
